# 3장 2강: SDK 개발 환경 구성과 연결 테스트
## 3. Supabase Client 초기화

In [1]:
import os
from supabase import create_client

# .env 파일의 값을 환경변수로 불러오기
url = os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_PUBLISHABLE_KEY')

# Client 초기화
supabase = create_client(url, key)

print(supabase)

## 4. 연결 테스트와 응답·오류 객체 확인

In [3]:
try:
    response = supabase.table("documents").select("*").limit(1).execute()
    print("연결 성공:", response.data)
except Exception as error:
    print("연결 실패:", error)

연결 성공: []


# 3장 3강: SDK 기반 데이터 생성 구현

## 1. SDK insert 구조

In [ ]:
response = supabase.table("users").insert({
    'email' : 'user03@test.org'
}).execute()

In [ ]:
print(response.data)

In [ ]:
response = supabase.table("documents").insert({
    "title": "두 번째 문서",
    "content": "본문 내용입니다.",
    'user_id': 'dacf75f4-43cd-483b-9c87-48bead6f8913'
}).execute()

print(response.data)

[{'id': '5c06b660-b7c7-4eba-b46f-4c8533bc169a', 'title': '두 번째 문서', 'content': '본문 내용입니다.', 'user_id': 'dacf75f4-43cd-483b-9c87-48bead6f8913', 'status': 'draft', 'created_at': '2026-09-09T02:38:01.618452+00:00'}]


In [22]:
# 회원 목록 조회
response = supabase.table("users").select("*").execute()
response.data

[{'id': '986ac33a-7812-40b0-8ab2-fbac7029893c',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:56.626338'},
 {'id': '1315560a-91f1-4707-bf2f-c7783b8be7e7',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:43:16.824917'},
 {'id': 'dbe1c643-ed1c-4156-b1f9-7fe645ea913f',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:46.41701'},
 {'id': '00b5a42c-743f-41bc-a480-39a19141141b',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:56:03.146712'},
 {'id': '8114d248-50d1-4091-af6b-39dc8bebf343',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:58:38.296372'},
 {'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:58:45.488342'},
 {'id': '7d45522b-a6dc-4c61-b985-6b1546c0c001',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:59:14.087146'}]

In [ ]:
response = (
    supabase.table("users")
    .select('*')
    .ilike('email', "user04%")
    .single() # 무조건 한개가 나와야한다. 반환값은 딕셔너리
    .execute()
)

#id, email, created_at = response.data[0]
#print(f"id={id}, email={email}, created_at={created_at}")

response.data

APIError: {'message': 'Cannot coerce the result to a single JSON object', 'code': 'PGRST116', 'hint': None, 'details': 'The result contains 0 rows'}

In [ ]:
response = (
    supabase.table("users")
    .select('*')
    .ilike('email', "user%")
    .limit(1)
    .offset(1) # 1인덱스에서 1개를 조회
    .maybe_single() # 0~1개, 0개이면 none
    .execute()
)

print(response)

TypeError: BaseSelectRequestBuilder.order() missing 1 required positional argument: 'column'

In [52]:
response = (
    supabase.table("users")
    .select('*')
    .order("created_at", desc=True)
    .execute()
)

response.data

[{'id': '7d45522b-a6dc-4c61-b985-6b1546c0c001',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:59:14.087146'},
 {'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:58:45.488342'},
 {'id': '8114d248-50d1-4091-af6b-39dc8bebf343',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:58:38.296372'},
 {'id': '00b5a42c-743f-41bc-a480-39a19141141b',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:56:03.146712'},
 {'id': 'dbe1c643-ed1c-4156-b1f9-7fe645ea913f',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:55:46.41701'},
 {'id': '1315560a-91f1-4707-bf2f-c7783b8be7e7',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:43:16.824917'},
 {'id': '986ac33a-7812-40b0-8ab2-fbac7029893c',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:56.626338'}]

In [55]:
# left join
response = (
    supabase.table("documents")
    .select("id, title, content, users(id, email)")
    .execute()
)

response.data

[{'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'title': '첫 번째 문서',
  'content': '본문 내용입니다.',
  'users': None},
 {'id': '5c06b660-b7c7-4eba-b46f-4c8533bc169a',
  'title': '두 번째 문서',
  'content': '본문 내용입니다.',
  'users': {'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
   'email': 'user03@test.org'}}]

In [57]:
# inner join(!inner)
response = (
    supabase.table("documents")
    .select("id, title, content, users!inner(id, email)")
    .execute()
)

response.data

[{'id': '5c06b660-b7c7-4eba-b46f-4c8533bc169a',
  'title': '두 번째 문서',
  'content': '본문 내용입니다.',
  'users': {'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
   'email': 'user03@test.org'}}]

## 2. 입력값과 DB 컬럼 매핑

## 3. 생성 결과와 오류 응답 처리

In [ ]:
create_document(None, "본문 내용입니다.", 1)

# 3장 4강: SDK 기반 데이터 조회 구현

## 1. SDK select 구조

## 2. 조건 필터링: eq, order, limit

## 3. 사용자별 조회와 결과 리스트 처리

# 3장 5강: SDK 기반 데이터 수정·삭제 구현

## 1. SDK update 구조

In [58]:
response = (
    supabase.table("documents")
    .select('*')
    .execute()
)

response.data

[{'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'title': '첫 번째 문서',
  'content': '본문 내용입니다.',
  'user_id': None,
  'status': 'draft',
  'created_at': '2026-09-09T02:37:46.206013+00:00'},
 {'id': '5c06b660-b7c7-4eba-b46f-4c8533bc169a',
  'title': '두 번째 문서',
  'content': '본문 내용입니다.',
  'user_id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'status': 'draft',
  'created_at': '2026-09-09T02:38:01.618452+00:00'}]

In [61]:
response = (
    supabase.table("documents")
    .update({
        "title" : "두 번째 문서(수정)"
    })
    .eq("id", "5c06b660-b7c7-4eba-b46f-4c8533bc169a")
    .execute()
)

response.data

[{'id': '5c06b660-b7c7-4eba-b46f-4c8533bc169a',
  'title': '두 번째 문서(수정)',
  'content': '본문 내용입니다.',
  'user_id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'status': 'draft',
  'created_at': '2026-09-09T02:38:01.618452+00:00'}]

## 2. SDK delete 구조

In [ ]:
response = (
    supabase.table("documents")
    .delete()
    .eq("id", "5c06b660-b7c7-4eba-b46f-4c8533bc169a")
    .execute()
)

response = (
    supabase.table("documents")
    .select('*')
    .execute()
)

[{'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913',
  'title': '첫 번째 문서',
  'content': '본문 내용입니다.',
  'user_id': None,
  'status': 'draft',
  'created_at': '2026-09-09T02:37:46.206013+00:00'}]

In [68]:
response.data

[]

## 3. 수정·삭제 후 검증과 오류 처리

In [69]:
response = (
    supabase.table("documents")
    .update({
        "status" : "published"
    })
    .eq("id", "dacf75f4-43cd-483b-9c87-48bead6f8913")
    .execute()
)

In [71]:
# 검증: 상태가 잘 바뀌었는지 다시 조회
check = supabase.table("documents").select("id, status").eq("id", "dacf75f4-43cd-483b-9c87-48bead6f8913").execute()
print(check.data)

[{'id': 'dacf75f4-43cd-483b-9c87-48bead6f8913', 'status': 'published'}]


# 3장 6강: Supabase Authentication 기본 흐름

## 2. 이메일 기반 회원가입과 로그인

In [80]:
key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(url, key)

"""
supabase.auth : 인증(로그인) / 인가(접근 제한) 관련 메서드 가지고 있는 객체
"""
#response = supabase.auth.sign_up({
#    "email" : "user02@test.org",
#    "password" : "password1234",
#    "phone" : "01010001000"
#})

'\nsupabase.auth : 인증(로그인) / 인가(접근 제한) 관련 메서드 가지고 있는 객체\n'

In [ ]:
response

In [86]:
# 로그인
response = supabase.auth.sign_in_with_password({
    "email" : "user01@test.org",
    "password" : "password1234"
})

response.user

User(id='d258ed8d-2bc5-4f62-b69e-2048711c534f', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'd258ed8d-2bc5-4f62-b69e-2048711c534f'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 40, 50, 831860, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 50, 886861, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 40, 50, 886861, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 49, 34, 277909, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 49, 34, 294015, tzinfo=TzInfo(0)), identities=[UserIdentity(id='d258ed8d-2bc5-4f62-b69e-2048711c534f', identity_id='e448749c-4145-

## 3. 세션, 로그아웃, user 객체

In [ ]:
# 현재 로그인된 사용자 확인


In [ ]:
# 로그아웃


## 4. 사용자 ID와 테이블 데이터 연결

# 3장 7강: 사용자별 데이터 접근과 권한 관리

## 2. user_id 기반 필터링

In [ ]:


# 본인 문서만 조회


# 3장 8강: SDK 기반 미니 과제 - 사용자별 문서 CRUD

## 2. Auth와 user_id 연결

In [ ]:

# 생성: 로그인 사용자의 문서


# 조회: 본인 문서만


## 3. 응답·오류 처리와 코드 구조 정리